Author: Chau Ha (Pearl)

Branch: team-02/chau-ha-d3

Target Dataset: American Airlines Passenger & Revenue Forecasting (AA_PAX_FORECASTING.csv) & Airport Info (AA_AIRPORT_INFO.csv)

1. Shape and Types
What we're looking at: Taking a first look at the files, our main passenger forecasting dataset (AA_PAX_FORECASTING.csv) sits at 1,533,522 rows across 8 columns, and the airport reference table (AA_AIRPORT_INFO.csv) has 4,565 rows and 18 columns.

Key columns: We have our temporal markers (DEP_MONTH, DEP_YEAR), route identifiers (ORG_AIRPRT, DES_AIRPRT), passenger demand (OD_PAX), revenue (OD_REV_USD), and system metadata (SF_LOAD_TMS).

import pandas as pd

df_pax = pd.read_csv('../../data/raw/AA_PAX_FORECASTING.csv')
df_airport = pd.read_csv('../../data/raw/AA_AIRPORT_INFO.csv')

print("Pax dataset missing values:\n", df_pax.isnull().sum())
print("\nAirport dataset missing values:\n", df_airport.isnull().sum())

null_states = df_airport[df_airport['STATE_PROVNC_CD'].isnull()]
print("\nSample international airports with null state codes:")
print(null_states[['AIRPRT_CD', 'CNTRY_CD']].head(10))

print("Correlation between OD_PAX and OD_REV_USD:")
print(df_pax[['OD_PAX', 'OD_REV_USD']].corr())

Takeaway: Therefore, we should cast our year and month fields to integers/categoricals, keep the airport codes ready for merging with our geographic metadata, and treat our passenger and revenue numbers as continuous floats.

2. Missingness (Deep-Dive Focus)
What the data shows: Running a null check across the main forecasting dataset gave us a clean slate—zero missing values across all 1.5 million rows! However, in the auxiliary airport reference table (AA_AIRPORT_INFO.csv), we do see missing values in secondary administrative columns like STATE_PROVNC_CD (2,383 missing) and CNTRY_CD (8 missing).


Systematic vs. Random: Those missing state codes aren't random noise; they're completely structural. They pop up precisely on international airports outside the US and Canada where state or province subdivisions don't apply.



Takeaway: Therefore, since the core forecasting dataset is complete, we don't need any imputation for our main routing features. For the airport reference table, instead of dropping those international rows, we should explicitly fill/flag missing state-province codes as 'INTL' so we don't lose valuable global airport data.

3. Distributions and Outliers
What the data shows: Our demand metrics (OD_PAX and OD_REV_USD) are heavily right-skewed. We have everything from low-frequency regional hops up to massive hub trunk routes (max passenger counts over 140,000 and revenues soaring past $121 million). These extreme values aren't data entry errors—they're real, high-volume hub-to-hub corridors (like DFW to ORD or MIA).

Takeaway: Therefore, we should apply a log-transformation ($\log(1 + x)$) to tame the skewness on volume and revenue during feature engineering, and definitely keep those high-volume hub routes instead of treating them as junk outliers.

4. Leakage Risk
What the data shows: Most of our inputs (DEP_MONTH, DEP_YEAR, origin/destination codes) are forward-looking schedule details we'd actually have at prediction time. But OD_REV_USD (total route revenue) is a classic trap—it's only fully realized after the flight happens and shares a tight, deterministic link with passenger volume. Using it as a feature would hand us massive target leakage.

Takeaway: Therefore, we must strictly drop financial realization metrics like OD_REV_USD from our feature set to ensure we're only feeding our model ex-ante information.

5. Relationships and Model Family View
What the data shows: Passenger demand doesn't move in a straight line; it's driven by tricky non-linear interactions, seasonal spikes across months, and network density across specific airport pairs. A standard linear regression model is going to miss all of those network synergies. Tree-based ensembles (like XGBoost or LightGBM) handle high-cardinality categorical airport codes and non-linear thresholds naturally without needing messy one-hot encoding matrices.

Takeaway: Therefore, our EDA points us straight toward Gradient Boosted Decision Trees (XGBoost / LightGBM) as our go-to model family, backed up by robust time-based validation.

6. Deep-Dive Hand-Off Table

| Column / feature | Decision (keep / drop / engineer / leak / id) | Reason |
|---|---|---|
| DEP_MONTH | keep / engineer | Essential temporal feature; engineer into cyclical sine/cosine components. |
| DEP_YEAR | keep | Captures macro-level multi-year secular demand trends and post-2020 recovery. |
| ORG_AIRPRT | keep / engineer | Core categorical route identifier; encode via target or frequency encoding. |
| DES_AIRPRT | keep / engineer | Core categorical destination identifier; encode via target or frequency encoding. |
| OD_PAX | keep | Primary target variable representing origin-destination passenger volume. |
| OD_REV_USD | leak / drop | Financial realization metric known post-flight; causes severe target leakage. |
| SF_LOAD_TMS | drop | System ingestion metadata irrelevant to flight passenger demand. |
| Unnamed: 0 | drop | Unnamed index artifact from CSV serialization; contains no predictive signal. |


7. Model-Family View (Summary in Writing)
Looking closely at the data, passenger demand across American Airlines' network is shaped by non-linear seasonal cycles, route density, and categorical airport pairings. Because tree-based gradient boosting frameworks excel at capturing complex feature interactions and high-cardinality categorical splits without forcing manual polynomial expansions, our team's modeling efforts should focus squarely on XGBoost regression.

Note on Encoding: If we use target encoding for high-cardinality airport codes (ORG_AIRPRT, DES_AIRPRT), it must be fit strictly on the training months within each cross-validation fold to prevent target leakage. Frequency encoding is a safer alternative that avoids folding leakage risk entirely.


Note on Missing Rows: Although the dataset has zero null entries, running a route-month check reveals that a small number of routes do not have records for every single month (indicating intermittent service or suspended routes rather than pure data gaps).

In [ ]:
import pandas as pd

# 1. Load datasets
df_pax = pd.read_csv('../../data/raw/AA_PAX_FORECASTING.csv')
df_airport = pd.read_csv('../../data/raw/AA_AIRPORT_INFO.csv')

print('--- Passenger Dataset ---')
print(df_pax.info())
print('\nMissing values:\n', df_pax.isnull().sum())

print('\n--- Airport Reference Table ---')
print(df_airport.info())
print('\nMissing values:\n', df_airport.isnull().sum())

# Verify missing state codes belong to international airports outside US/Canada
null_states = df_airport[df_airport['STATE_PROVNC_CD'].isnull()]
print('\nSample international airports with null state codes:')
print(null_states[['AIRPRT_CD', 'CNTRY_CD']].head(10))

# 2. Leakage Check (OD_REV_USD)
print('\nCorrelation between OD_PAX and OD_REV_USD:')
print(df_pax[['OD_PAX', 'OD_REV_USD']].corr())

# 3. Missing Rows Check (Route-Month Completeness)
total_months = df_pax['DEP_MONTH'].nunique()
route_months = df_pax.groupby(['ORG_AIRPRT', 'DES_AIRPRT'])['DEP_MONTH'].nunique()
incomplete_routes = route_months[route_months < total_months]
print(f'\nTotal unique routes: {len(route_months)}')
print(f'Routes with incomplete monthly records: {len(incomplete_routes)}')